In [1]:
# %pip install python-dotenv
# %uv add dspy
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

## check aicodetools library

In [4]:


import time
import threading
import tiktoken
from collections import deque
import dspy
from dspy.utils.callback import BaseCallback


class SlidingWindowLimiter:
    """Rate limiter that enforces both request and token limits per rolling minute."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, max_requests_per_min=1000, max_tokens_per_min=2_000_000):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.max_requests = max_requests_per_min
            cls._instance.max_tokens = max_tokens_per_min
            cls._instance.requests = deque()  # [(timestamp, tokens)]
            cls._instance._lock = threading.Lock()
            cls._instance.encoder = tiktoken.get_encoding("cl100k_base")
        return cls._instance

    def _cleanup(self, now):
        """Remove entries older than 60s."""
        while self.requests and now - self.requests[0][0] > 60:
            self.requests.popleft()

    def _count(self):
        """Total requests & tokens in current 60s window."""
        total_tokens = sum(t for _, t in self.requests)
        return len(self.requests), total_tokens

    def acquire(self, tokens_used=0):
        """Wait until request fits in sliding 60s window."""
        with self._lock:
            while True:
                now = time.time()
                self._cleanup(now)
                req_count, token_count = self._count()

                # Can fit in current 60s window?
                if (req_count < self.max_requests and
                        token_count + tokens_used <= self.max_tokens):
                    # Record the new request
                    self.requests.append((now, tokens_used))
                    break  # proceed

                # Otherwise, figure out when we can retry
                oldest_time = self.requests[0][0]
                sleep_time = max(0.01, 60 - (now - oldest_time))
                print(f"⚠️ Throttling: sleeping {sleep_time:.2f}s (req={req_count}, tokens={token_count})")
                time.sleep(sleep_time)


class DelayAndLogCallback(BaseCallback):
    """DSPy callback using sliding window limiter."""

    def __init__(self):
        self.limiter = SlidingWindowLimiter()

    def _estimate_tokens(self, messages=None, prompt=None):
        """Estimate token usage using tiktoken."""
        text = ""
        if prompt:
            text = str(prompt)
        elif messages:
            # concatenate all message contents
            text = " ".join(m.get("content", "") for m in messages)
        return len(self.limiter.encoder.encode(text))

    def on_lm_start(self, *args, **kwargs):
        inputs = kwargs.get("inputs") or {}
        prompt = inputs.get("prompt")
        messages = inputs.get("messages")
        tokens_used = self._estimate_tokens(messages=messages, prompt=prompt)
        self.limiter.acquire(tokens_used=tokens_used)

    def on_lm_end(self, *args, **kwargs):
        pass



In [5]:

# from aicodetools import ClientManager 

# code_tool_manager = ClientManager(
#                 "super-bench:latest", base_log_dir="runs/super/"
#             )

# code_tool_client = code_tool_manager.get_client('initial')
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
import dspy

In [6]:


lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=30, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)



# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']
## Load the benchmark and view one example from the benchmark


['This is a test!']
['This is a test!']


In [7]:
from gepa_artifact.benchmarks.super_bench.super_utils import FinishResponse
from gepa_artifact.benchmarks.super_bench import benchmark_with_gold as sb_metas


bench = sb_metas[0].benchmark(with_gold=True)
len(bench.train_set), len(bench.val_set), len(bench.test_set)


import pprint
pprint.pprint(bench.train_set[0])



## Load the program and display the program
# The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively
program = sb_metas[0].program[0]
program


Available Tools for e4743024-a401-499b-ae1c-73d791800572: on runtime aicodetools-e4743024-a401-499b-ae1c-73d791800572-1d8d89a9 4
Available Tools for 5a678a94-b728-446e-98c6-266cb06cdc00: on runtime aicodetools-5a678a94-b728-446e-98c6-266cb06cdc00-059510a2 4
Example({'instance_id': 'pie-perf', 'github_repo': 'https://github.com/madaan/pie-perf', 'git_commit': 'ee1989b66756470622e3b89c4aa031f083f57ef9', 'query': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0). Once evaluated, report the result problem_id and input_acc for each problem of the dataset, as a json list of dictionaries structured as follows: [{"problem_id": "", "input_acc": 0.0}] (replace "" and 0.0 with the actual values).\n\nAdditional instructions:\n1. Set "num_trials": 2 in the evaluation configuration file to reduce computation time.\n2. Load only the first 10 rows of the dataset.\n\nGit repository: ht

react.react = Predict(StringSignature(query, github_repo, git_commit, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions='Solve the question and provide the answer in the correct format.\n\nYou are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen selecting the next_tool_name and its next_tool_args, the tool must be one of:\n\n(1) read_file, whose description is <desc>          Read file with optional line range and/or 

In [10]:

import dspy
from gepa_artifact.gepa.gepa import GEPA,GEPAState
from gepa_artifact.utils.capture_stream_logger import Logger

import time

In [ ]:


### Make Sure docker is installed and running
## Define an evaluator and evaluate the base program




runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)
gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))



if sb_metas[0].feedback_fn_maps is None or sb_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = sb_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = sb_metas[0].feedback_fn_maps[0]



optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=sb_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    num_iters=9,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=9)
## Optimize the program with GEPA



In [36]:
state = GEPAState.load('runs/gepa-state-with-gold')

In [37]:
state.num_full_ds_evals

9

In [40]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)


gepa_state = state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_progs = gepa_state.program_candidates
best_prog = best_progs[best_prog_idx]


optimized_program = best_prog
latest_prog = best_progs[-1]

In [41]:

### Let's print the prompts that GEPA discovered
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: react.react
Prompt:
Solve the question and provide the answer in the correct format.

You are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.
Your goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.

To do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.
After each tool call, you receive a resulting observation, which gets appended to your trajectory.

When writing next_thought, you may reason about the current situation and plan for future steps.
When selecting the next_tool_name and its next_tool_args, the tool must be one of:

(1) read_file, whose description is <desc>          Read file with optional line range and/or regex filtering.            Args:              file_path: Absolute path to file (e.g., /workspace/main.py)              lines_s

In [42]:

sb_metas[0].program[0].get_lm()

In [43]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='optimized_react.json',
    save_as_csv='optimized_react.csv'
)

In [44]:
results = evaluate(optimized_program)

  0%|          | 0/27 [00:00<?, ?it/s]Available Tools for g-transformer: on runtime aicodetools-g-transformer-04bdb0b4 4
Available Tools for spa: on runtime aicodetools-spa-be92a78f 4
Available Tools for mezo: on runtime aicodetools-mezo-e1bb7092 4
Available Tools for mode-connectivity-plm: on runtime aicodetools-mode-connectivity-plm-6cba4727 4
Available Tools for mbib: on runtime aicodetools-mbib-3375316d 4
Available Tools for unsupervisedhierarchicalsymbolicregression: on runtime aicodetools-unsupervisedhierarchicalsymbolicregression-a9293019 4
Available Tools for conv_graph: on runtime aicodetools-conv_graph-51401888 4
Available Tools for pira: on runtime aicodetools-pira-b99b0549 4
Available Tools for pet: on runtime aicodetools-pet-2ed8044a 4
Cleaned up Tools spa : True  True
success=False structured_output=None reasoning="The requested experiment cannot be run: the repository does not contain the 'alpaca_data_en_52k' dataset, nor any pipelines or scripts for training the SPA mod

Cleaned up Tools mode-connectivity-plm : True  True
success=False structured_output={'eval_loss': None} reasoning='Due to multiple workspace or environment resets, all previously written scripts and cloned repository files have been erased. The essential finetuning script, configuration file (mnli.json), and RoBERTa_model folder are missing and cannot be accessed, making it impossible to run the experiment and report the required eval loss as requested.' summary='Attempted to set up the environment, installed required packages, cloned the repo, and wrote a finetuning script for roberta-base on Rotten Tomatoes with first 10 rows per split using mnli.json hyperparameters. However, workspace resets continually removed all relevant files and scripts; thus, the experiment could not be executed and no metrics could be reported.'
The experiment could not be completed due to workspace/environment resets that resulted in the loss of all required files, including the cloned `thunlp/mode-connecti

Cleaned up Tools team : False  False
success=False structured_output=None reasoning="All environment setup, data formatting, dependency installation, and model selection were completed successfully. However, TEAM's training script requires a CUDA-enabled NVIDIA GPU and fails to initialize if run in a CPU-only environment. This precluded training and evaluation, so no metrics could be extracted. To complete this task, run the code in a GPU-enabled environment." summary='Repository cloned, dataset downloaded/transformed, dependencies installed and compatibility issues resolved. All code paths are ready for GPU training. Training failed due to lack of NVIDIA driver/GPU; run in a GPU environment to produce valid TEAM metrics.'
To run TEAM training/evaluation on your custom multi-choice dataset, I performed extensive environment setup and debugged repeated compatibility issues:

1. Cloned the TEAM repository at the requested commit.
2. Downloaded your dataset from Google Drive, inspected it

2025/11/02 15:30:06 INFO dspy.evaluate.evaluate: Average Metric: 0.8333333333333333 / 27 (3.1%)


,instance_id,github_repo,git_commit,query,query_components,solution_dependencies,answer,landmarks,solution,trajectory,reasoning,result,super_score
0,g-transformer,https://github.com/baoguangsheng/g-transformer,dcc7695ceb0ecc3250e1c28215e9ddcd22700b39,Use the https://github.com/baoguangsheng/g-transformer repository ...,{'e2e_task': 'Use the https://github.com/baoguangsheng/g-transform...,absl-py==1.4.0 aiohttp==3.9.5 aiosignal==1.3.1 alabaster==0.7.16 a...,"{""Sentence-level BLEU"": 0.0, ""Document-level BLEU"": 0.01}",['INFO\\] Building segmented data' 'INFO \\| fairseq_cli.preproces...,"[{'action': {'content': '# ## Solution', 'type': 'execute'}, 'obse...",{'thought_0': 'I need to determine how fine-tuning is performed in...,The task could not be completed due to persistent environment erro...,"success=False structured_output={'Sentence-level BLEU': 0.0, 'Docu...","✔️ [Prediction(\n score=0.25,\n score_dict={'submitted': 1, ..."
1,spa,https://github.com/OceannTwT/SPA,a8bb190a182c076f80d90ac59921abd1df29b0ae,"Train the SPA model on alpaca_data_en_52k (from the repo), startin...",{'e2e_task': 'Train the SPA model on alpaca_data_en_52k (from the ...,absl-py==1.4.0 accelerate==0.30.0 aiohttp==3.9.5 aiosignal==1.3.1 ...,"{""training_loss"": 4.97342586517334}","[Generating train split: \d+ examples, >> \*\*\*\*\* Running train...","[{'action': {'content': '# ### Setup', 'type': 'execute'}, 'observ...","{'thought_0': 'To follow the instructions, I first need to inspect...","Despite thoroughly reviewing the repository structure, training sc...","success=False structured_output=None reasoning=""The requested expe...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 0, '..."
2,mezo,https://github.com/princeton-nlp/MeZO,552cb1b710767f9a6e1dc8f9645d7640376f9941,"Train using the ""MeZO"" method (no prefix-tuning / lora) on the RTE...","{'e2e_task': 'Train using the ""MeZO"" method (no prefix-tuning / lo...",absl-py==1.4.0 accelerate==0.30.0 aiohttp==3.9.5 aiosignal==1.3.1 ...,"{""accuracy"": 0.8, ""dev_accuracy"": 0.4}",['- INFO - Sample train set \\d+/\\d+' '- INFO - \\*\\*\\*\\*\\* R...,"[{'action': {'content': '# ## Solution', 'type': 'execute'}, 'obse...","{'thought_0': 'To train on RTE with MeZO\'s method, we need to loc...",The experiment could not be executed due to persistent workspace i...,success=False structured_output={} reasoning='The experiment is bl...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 0, '..."
3,mode-connectivity-plm,https://github.com/thunlp/mode-connectivity-plm,f45efd03fabdb23777c3ddcb35612cb1f7b1960c,"Fine-tune roberta-base on a single-task, ""rotten tomatoes"" which i...","{'e2e_task': 'Fine-tune roberta-base on a single-task, ""rotten tom...",absl-py==1.4.0 accelerate==0.30.0 aiohttp==3.9.5 aiosignal==1.3.1 ...,"{""eval_loss"": 0.0368}","[Running tokenizer on dataset:, \*\*\*\*\* train metrics \*\*\*\*\...","[{'action': {'content': '# ## Task instructions\n# \n# ""Fine-tune ...","{'thought_0': ""To use the hyperparameters for fine-tuning, I need ...",The experiment could not be completed due to workspace/environment...,success=False structured_output={'eval_loss': None} reasoning='Due...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
4,mbib,https://github.com/Media-Bias-Group/MBIB,b9a887ffd461fa462e89835fc27b36e370091954,"Train a bart-base model on the ""linguistic-bias"" task using the ba...","{'e2e_task': 'Train a bart-base model on the ""linguistic-bias"" tas...",absl-py==1.4.0 accelerate==0.30.0 aiohttp==3.9.5 aiosignal==1.3.1 ...,"{""average_weighted_f1"": 0.44272727272727275}","[Training Initialized for fold \d+, The current dev loss: tensor\(...","[{'action': {'content': '# ## Solution', 'type': 'execute'}, 'obse...","{'thought_0': ""To proceed, I need to locate the generated `linguis...","Despite successful cloning of the MBIB repository, environmental o...",success=False structured_output={} reasoning='Workspace mounting a...,"✔️ [Prediction(\n score=0.0,\n 

Cleaned up Tools dpt : False  False
success=False structured_output={'accuracy': None} reasoning='All environment, code compatibility, and script patches were completed successfully: dependencies aligned, MRPC data loaded, code and devices patched for CPU, and legacy transformers pinned. However, training and evaluation could not be completed due to persistent server connection errors at the final run step. Please re-run the script when server access resumes to obtain the required accuracy metric.' summary='Environment and code fully prepared for decomposed prompt tuning of t5-small on MRPC for 1 epoch, with all dependency and device issues fixed (patched for CPU and legacy transformers API). The final experiment execution was blocked by a server error; once resolved, running the prescribed command will yield the development set accuracy for reporting.'
1. Environment setup: Cloned the `dpt` repository at the required commit, confirmed dataset structure, and examined all necessary trai

Cleaned up Tools galore : False  False
success=False structured_output={} reasoning='Environment was fully prepared: repository cloned and patched, dependencies installed, script fixed for telemetry API removal and sample restriction. Multiple infrastructure-level issues prevented execution of the actual fine-tuning script and thus no accuracy or F1 metric could be extracted. All scientific setup steps were verified.' summary='1. Cloned galore repo at required commit. 2. Patched run_glue.py to restrict train/validation sets to 10 samples each. 3. Installed/verified all required libraries and resolved binary incompatibilities. 4. Removed problematic telemetry import from script. 5. Attempted to execute training, but server connection error prevented completion. Accurate metrics cannot be reported until environment issue is resolved.'
The experiment required fine-tuning the roberta-base model on MRPC using the GaLore optimizer, with only the first 10 samples of train/validation, for 1 ep

Cleaned up Tools dpt : False  False
success=False structured_output={} reasoning='Workflow setup, code fixes, and correct dataset/model arguments for decomposed prompt tuning on T5-small were completed. However, all attempts to run training, extract metrics, or even perform basic file access failed with persistent server/environment connectivity errors. These infrastructure issues prevented the fine-tuning run and metric extraction.' summary="1. Repository cloned and code inspected for required hyperparameters and dataset mapping.\n2. Minor indentation error in main_ddp.py was fixed.\n3. MRPC task string determined and correct training command assembled for 1-epoch prompt tuning.\n4. Actual training, evaluation, and metric extraction could not be performed due to repeated 'Failed to connect to server' errors, blocking all further progress."
The experiment workflow was blocked by repeated server/environment connection errors after all setup and code fixes. The correct repo, dataset, and

In [63]:
df

,idx,taskid,submitted,output_match,landmarks
0,1,spa,0.0,0.0,0.000000
1,2,mbib,0.0,0.0,0.000000
2,3,conv_graph,0.0,0.0,0.000000
3,4,blockskim,0.0,0.0,0.000000
4,5,mode-connectivity-plm,1.0,0.0,0.000000
5,6,mezo,0.0,0.0,0.000000
6,7,transnormerllm,0.0,0.0,0.000000
7,8,g-transformer,1.0,0.5,0.000000
8,9,unsupervisedhierarchicalsymbolicregression,0.0,0.0,0.000000
9,10,bert-lnl,0.0,0.0,0.000000
